### Tools

Models can request to call tools that perform task such as fetching data from database, searching the web, or running code, tools are pairing of:
1. A schema, including the name of the tool, a description and/or argument defination (often a JSON Schema)
2. A fuction or coroutine to execute.

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [15]:
from langchain.chat_models import init_chat_model
os.environ["openrouter_api_key"] = openrouter_api_key
model = init_chat_model(
    "openrouter:deepseek/deepseek-v3.2",)
response = model.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.


In [16]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tools = model.bind_tools([get_weather])

In [17]:
response = model_with_tools.invoke("What's the weather like in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

content='I' additional_kwargs={} response_metadata={'model_name': 'deepseek/deepseek-v3.2-20251201', 'id': 'gen-1781623150-09wAfZ5tUHiXAfKxcUOa', 'created': 1781623150, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019ed103-d703-7b53-aa6a-a9d927a2c7a9-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_54aa804799074e7091a338b6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 314, 'output_tokens': 57, 'total_tokens': 371, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}
tool: get_weather
args: {'location': 'New York'}


### Tool Execution Loop

In [18]:
# Step 1: Model generate the tool call
message = [{"role": "user", "content": "What's the weather like in New York?"}]
ai_msg = model_with_tools.invoke(message)
print(ai_msg)
message.append(ai_msg)

#step 2: Execute the tool and collect results
for tool_call in ai_msg.tool_calls:
    # execute the tool call and generate arguments
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# Step 3: Model generates final response
final_response = model_with_tools.invoke(message)
print(final_response.text)
# current weather

content="I'll check the current weather in New York for you.\n\n" additional_kwargs={} response_metadata={'model_name': 'deepseek/deepseek-v3.2-20251201', 'id': 'gen-1781623152-gO8npNhnxWHsHNhXOXmF', 'created': 1781623152, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019ed103-e019-7252-8ac7-c606626fd0ac-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_7dcab39b3e3948d6a160f447', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 310, 'output_tokens': 56, 'total_tokens': 366, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}
The current weather in New York is sunny with a high temperature of 25°C (77°F). It's a pleasant, warm day!


In [20]:
message


[{'role': 'user', 'content': "What's the weather like in New York?"},
 AIMessage(content="I'll check the current weather in New York for you.\n\n", additional_kwargs={}, response_metadata={'model_name': 'deepseek/deepseek-v3.2-20251201', 'id': 'gen-1781623152-gO8npNhnxWHsHNhXOXmF', 'created': 1781623152, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019ed103-e019-7252-8ac7-c606626fd0ac-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_7dcab39b3e3948d6a160f447', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 310, 'output_tokens': 56, 'total_tokens': 366, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}),
 ToolMessage(content='The current weather in New York is sunny with a high of 25°C.', name='get_weather', tool_call_id='call_7dcab39b3e3948d6a160f447')]